In [ ]:
# =============================================================
# NOTEBOOK 05 – RESIDUAL ANALYSIS & CLASSIFICATION
# Môn: Học Máy MAT3533
# Đề tài: Hồi quy – IBM HR Analytics Employee Attrition
# Thực hiện: Phan Thanh Lâm & Trịnh Quang Hải
# Tuần: 3 | Ngày T2 – T4
# =============================================================

# ─── MỤC TIÊU ───────────────────────────────────────────────
# 1.4b: Residual Analysis (trực quan hóa phần dư)
# 1.4c: Chuyển bài toán sang classification (3 mức)

# ─── IMPORTS ───────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import scipy.stats as stats
import os
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 150,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.linestyle': '--',
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

COLORS = {
    'blue': '#534AB7',
    'red': '#E24B4A',
    'green': '#1D9E75',
    'orange': '#EF9F27',
    'purple': '#534AB7',
    'teal': '#1D9E75',
    'coral': '#E24B4A',
}

# ─── PATHS ─────────────────────────────────────────────────
SCALED_PATH = '../data/processed/df_scaled.csv'
RESULTS_PATH = '../results'
os.makedirs(RESULTS_PATH, exist_ok=True)
os.makedirs('../outputs/figures', exist_ok=True)

print("=" * 60)
print("NOTEBOOK 05 – RESIDUAL ANALYSIS & CLASSIFICATION")
print("=" * 60)

# ═══════════════════════════════════════════════════════════
# BƯỚC 1 – LOAD DATA & BEST MODEL
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 1 – LOAD DATA & BEST MODEL")
print("=" * 60)

df_scaled = pd.read_csv(SCALED_PATH)
print(f"✓ Loaded df_scaled: {df_scaled.shape}")

TARGET = 'MonthlyIncome'
y = df_scaled[TARGET].values

# Load best model từ regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X_raw = df_scaled.drop(columns=[TARGET, 'Attrition'], errors='ignore')
X_train, X_test, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=42)

best_model = LinearRegression()
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print(f"✓ Best model: Linear Regression")
print(f"✓ Test size: 20% (294 samples)")

# ═══════════════════════════════════════════════════════════
# BƯỚC 2 – RESIDUAL ANALYSIS (1.4b)
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 2 – RESIDUAL ANALYSIS (1.4b)")
print("=" * 60)

residuals = y_test - y_pred

# Hình 16: Residual Analysis
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Hình 16: Residual Analysis', fontsize=15, fontweight='bold')

# 1. Residuals vs Fitted
axes[0, 0].scatter(y_pred, residuals, alpha=0.5, color=COLORS['blue'], edgecolors='white', linewidth=0.5)
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Fitted Values (USD)')
axes[0, 0].set_ylabel('Residuals (USD)')
axes[0, 0].set_title('Residuals vs Fitted')
axes[0, 0].grid(True, alpha=0.3)

# 2. Q-Q Plot
stats.probplot(residuals, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot')
axes[0, 1].set_xlabel('Theoretical Quantiles')
axes[0, 1].set_ylabel('Sample Quantiles')
axes[0, 1].grid(True, alpha=0.3)

# 3. Histogram of Residuals
axes[0, 2].hist(residuals, bins=30, edgecolor='black', color=COLORS['teal'], alpha=0.7)
axes[0, 2].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0, 2].set_xlabel('Residuals (USD)')
axes[0, 2].set_ylabel('Frequency')
axes[0, 2].set_title('Histogram of Residuals')
axes[0, 2].grid(True, alpha=0.3, axis='y')

# 4. Residuals vs Order
axes[1, 0].plot(range(len(residuals)), residuals, 'o-', alpha=0.4, color=COLORS['orange'], markersize=3)
axes[1, 0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1, 0].set_xlabel('Observation Order')
axes[1, 0].set_ylabel('Residuals (USD)')
axes[1, 0].set_title('Residuals vs Order')
axes[1, 0].grid(True, alpha=0.3)

# 5. Density Plot
sns.kdeplot(residuals, ax=axes[1, 1], fill=True, color=COLORS['purple'], alpha=0.5)
axes[1, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1, 1].set_xlabel('Residuals (USD)')
axes[1, 1].set_ylabel('Density')
axes[1, 1].set_title('Density of Residuals')
axes[1, 1].grid(True, alpha=0.3)

# 6. Boxplot
box = axes[1, 2].boxplot(residuals, patch_artist=True,
                          boxprops=dict(facecolor=COLORS['coral'], alpha=0.7),
                          medianprops=dict(color='black', linewidth=2),
                          flierprops=dict(marker='o', markersize=4, alpha=0.5))
axes[1, 2].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1, 2].set_xticklabels(['Residuals'])
axes[1, 2].set_ylabel('Residuals (USD)')
axes[1, 2].set_title('Boxplot of Residuals')
axes[1, 2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../outputs/figures/fig16_residual_analysis.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓ Saved: outputs/figures/fig16_residual_analysis.png")

# Kiểm định thống kê
shapiro_stat, shapiro_p = stats.shapiro(residuals[:5000] if len(residuals) > 5000 else residuals)
jb_stat, jb_p = stats.jarque_bera(residuals)

print(f"\n📊 Kiểm định phân phối chuẩn của residuals:")
print(f"  Shapiro-Wilk p-value: {shapiro_p:.6f}")
print(f"  Jarque-Bera p-value:  {jb_p:.6f}")
print(f"  Mean của residuals:   {np.mean(residuals):.4f} (≈0 → không bias)")
print(f"  Std của residuals:    {np.std(residuals):.2f} USD")
print(f"  Skewness:             {stats.skew(residuals):.4f}")
print(f"  Kurtosis:             {stats.kurtosis(residuals):.4f}")

# ═══════════════════════════════════════════════════════════
# BƯỚC 3 – CLASSIFICATION (1.4c)
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 3 – CLASSIFICATION (1.4c)")
print("=" * 60)

# Chia MonthlyIncome thành 3 khoảng bằng nhau (equal frequency)
from sklearn.preprocessing import StandardScaler

# Đọc lại df_cleaned để có target dạng gốc
df_cleaned = pd.read_csv('../data/processed/df_cleaned.csv')
y_original = df_cleaned[TARGET].values

# Chia 3 mức xấp xỉ bằng nhau
labels, bins = pd.qcut(y_original, q=3, labels=['Low', 'Medium', 'High'], retbins=True)

print(f"\n✓ Ngưỡng phân chia:")
print(f"  Low    : ≤ {bins[1]:,.0f} USD ({np.sum(labels == 'Low')} samples)")
print(f"  Medium : {bins[1]:,.0f} - {bins[2]:,.0f} USD ({np.sum(labels == 'Medium')} samples)")
print(f"  High   : ≥ {bins[2]:,.0f} USD ({np.sum(labels == 'High')} samples)")

# Chuẩn bị data cho classification
X_class = df_scaled.drop(columns=[TARGET, 'Attrition'], errors='ignore')
y_class = labels

# Split
X_train, X_test, y_train, y_test = train_test_split(X_class, y_class, test_size=0.2, random_state=42, stratify=y_class)

print(f"\n✓ Train set: {X_train.shape}")
print(f"✓ Test set:  {X_test.shape}")

# Models
classification_models = {
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42)
}

# Train và đánh giá
classification_results = []

for name, model in classification_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    # Tính macro avg cho precision, recall, f1
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    
    classification_results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Precision (macro)': round(report['macro avg']['precision'], 4),
        'Recall (macro)': round(report['macro avg']['recall'], 4),
        'F1 (macro)': round(report['macro avg']['f1-score'], 4)
    })

results_df = pd.DataFrame(classification_results)
print("\n📊 Kết quả Classification (dữ liệu gốc):")
print(results_df.to_string(index=False))

# Hình 17: Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Hình 17: Confusion Matrix cho Classification', fontsize=13, fontweight='bold')

for idx, name in enumerate(classification_models.keys()):
    model = classification_models[name]
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred, labels=['Low', 'Medium', 'High'])
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Low', 'Medium', 'High'],
                yticklabels=['Low', 'Medium', 'High'],
                cbar=False, square=True)
    axes[idx].set_title(f'{name}')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('../outputs/figures/fig17_confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓ Saved: outputs/figures/fig17_confusion_matrix.png")

# Hình 18: Classification Results Comparison
fig, ax = plt.subplots(figsize=(10, 6))
metrics = ['Accuracy', 'Precision (macro)', 'Recall (macro)', 'F1 (macro)']
x = np.arange(len(metrics))
width = 0.35

nb_vals = [results_df[results_df['Model'] == 'Naive Bayes'][m].values[0] for m in metrics]
lr_vals = [results_df[results_df['Model'] == 'Logistic Regression'][m].values[0] for m in metrics]

bars1 = ax.bar(x - width/2, nb_vals, width, label='Naive Bayes', color=COLORS['teal'], alpha=0.8)
bars2 = ax.bar(x + width/2, lr_vals, width, label='Logistic Regression', color=COLORS['orange'], alpha=0.8)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Metrics')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel('Score')
ax.set_title('Hình 18: So sánh Classification Models', fontweight='bold')
ax.legend(loc='lower right')
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../outputs/figures/fig18_classification_comparison.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓ Saved: outputs/figures/fig18_classification_comparison.png")

# ═══════════════════════════════════════════════════════════
# BƯỚC 4 – CLASSIFICATION TRÊN PCA 15 COMPONENTS
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 4 – CLASSIFICATION TRÊN PCA 15 COMPONENTS")
print("=" * 60)

from sklearn.decomposition import PCA

pca15 = PCA(n_components=15, random_state=42)
X_pca15 = pca15.fit_transform(X_class)
print(f"✓ PCA 15 components — variance explained: {pca15.explained_variance_ratio_.sum()*100:.1f}%")

X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(
    X_pca15, y_class, test_size=0.2, random_state=42, stratify=y_class
)

classification_results_pca = []

for name, model in classification_models.items():
    model.fit(X_train_pca, y_train_pca)
    y_pred_pca = model.predict(X_test_pca)
    acc = accuracy_score(y_test_pca, y_pred_pca)
    report = classification_report(y_test_pca, y_pred_pca, output_dict=True, zero_division=0)
    classification_results_pca.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'Precision (macro)': round(report['macro avg']['precision'], 4),
        'Recall (macro)': round(report['macro avg']['recall'], 4),
        'F1 (macro)': round(report['macro avg']['f1-score'], 4)
    })

results_pca_df = pd.DataFrame(classification_results_pca)
print("\n📊 Kết quả Classification (PCA 15 components):")
print(results_pca_df.to_string(index=False))

# Hình 17b: Confusion Matrix PCA
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Hình 17b: Confusion Matrix — PCA 15 Components', fontsize=13, fontweight='bold')

for idx, name in enumerate(classification_models.keys()):
    model = classification_models[name]
    y_pred_pca = model.predict(X_test_pca)
    cm = confusion_matrix(y_test_pca, y_pred_pca, labels=['Low', 'Medium', 'High'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=axes[idx],
                xticklabels=['Low', 'Medium', 'High'],
                yticklabels=['Low', 'Medium', 'High'],
                cbar=False, square=True)
    axes[idx].set_title(f'{name} (PCA 15)')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('../outputs/figures/fig17b_confusion_matrix_pca.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓ Saved: outputs/figures/fig17b_confusion_matrix_pca.png")

# Hình 18b: So sánh Raw vs PCA
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Hình 18b: Raw vs PCA 15 — Classification Comparison', fontsize=13, fontweight='bold')

metrics = ['Accuracy', 'Precision (macro)', 'Recall (macro)', 'F1 (macro)']
x = np.arange(len(metrics))
width = 0.35

for ax, model_name in zip(axes, ['Naive Bayes', 'Logistic Regression']):
    raw_vals = [results_df[results_df['Model'] == model_name][m].values[0] for m in metrics]
    pca_vals = [results_pca_df[results_pca_df['Model'] == model_name][m].values[0] for m in metrics]
    bars1 = ax.bar(x - width/2, raw_vals, width, label='Raw Data', color=COLORS['teal'], alpha=0.8)
    bars2 = ax.bar(x + width/2, pca_vals, width, label='PCA 15', color=COLORS['orange'], alpha=0.8)
    for bar in list(bars1) + list(bars2):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(['Acc', 'Prec', 'Rec', 'F1'])
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.15)
    ax.set_title(model_name)
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../outputs/figures/fig18b_raw_vs_pca_classification.png', bbox_inches='tight', dpi=150)
plt.close()
print("✓ Saved: outputs/figures/fig18b_raw_vs_pca_classification.png")

# Lưu results (gộp Raw + PCA)
raw_df = pd.DataFrame(classification_results); raw_df['Data'] = 'Raw'
pca_df = results_pca_df.copy(); pca_df['Data'] = 'PCA_15'
all_clf = pd.concat([raw_df, pca_df], ignore_index=True)
all_clf.to_csv(f'{RESULTS_PATH}/classification_results.csv', index=False)
print(f"✓ Saved: {RESULTS_PATH}/classification_results.csv")

# ═══════════════════════════════════════════════════════════
# TÓM TẮT CUỐI
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("📊 TÓM TẮT NOTEBOOK 05")
print("=" * 60)
print(f"""
  Residual Analysis:
    - Mean residuals: {np.mean(residuals):.4f} (≈0)
    - Std residuals:  {np.std(residuals):.2f} USD
    - Shapiro p-value: {shapiro_p:.6f}
  
  Classification (3 mức):
    - Ngưỡng: ≤{bins[1]:,.0f} | {bins[1]:,.0f}-{bins[2]:,.0f} | ≥{bins[2]:,.0f}
    - Best model: {'Logistic Regression' if lr_vals[0] > nb_vals[0] else 'Naive Bayes'}
    - Best accuracy: {max(lr_vals[0], nb_vals[0])*100:.1f}%
  
  Output files:
    📁 outputs/figures/fig16_residual_analysis.png
    📁 outputs/figures/fig17_confusion_matrix.png
    📁 outputs/figures/fig18_classification_comparison.png
    📁 results/classification_results.csv
""")
print("=" * 60)
print("✅ notebook_05_residual_classification - HOÀN THÀNH!")

NOTEBOOK 05 – RESIDUAL ANALYSIS & CLASSIFICATION

BƯỚC 1 – LOAD DATA & BEST MODEL
✓ Loaded df_scaled: (1470, 48)
✓ Best model: Linear Regression
✓ Test size: 20% (294 samples)

BƯỚC 2 – RESIDUAL ANALYSIS (1.4b)
✓ Saved: outputs/figures/fig16_residual_analysis.png

📊 Kiểm định phân phối chuẩn của residuals:
  Shapiro-Wilk p-value: 0.000015
  Jarque-Bera p-value:  0.000000
  Mean của residuals:   58.6692 (≈0 → không bias)
  Std của residuals:    1167.74 USD
  Skewness:             0.6039
  Kurtosis:             1.2459

BƯỚC 3 – CLASSIFICATION (1.4c)

✓ Ngưỡng phân chia:
  Low    : ≤ 3,632 USD (490 samples)
  Medium : 3,632 - 6,529 USD (490 samples)
  High   : ≥ 6,529 USD (490 samples)

✓ Train set: (1176, 46)
✓ Test set:  (294, 46)

📊 Kết quả Classification (dữ liệu gốc):
              Model  Accuracy  Precision (macro)  Recall (macro)  F1 (macro)
        Naive Bayes    0.6497             0.7191          0.6497      0.6338
Logistic Regression    0.8197             0.8356          0.8197 